In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.signal as sci_sig
from flax import nnx

from exp_mpc.etc import ref_filt
from exp_mpc.etc import trip

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
test_file_path = "/Users/jozbee/work/eng/comp/data/clean_00_sms_drive.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
test_acc_ref, test_omega_ref = load_clean_references(test_file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)
test_acc_ref = jnp.clip(test_acc_ref, -1.0, 1.0)

In [ ]:
# data_range = [0, 90 * 200]
# data_range = [0, 4000]
# data_range = [8000, 12000]
# data_range = [8000, 90 * 200]
# data_range = [8000, 150 * 200]
data_range = [0, 250 * 200]
# data_range = [0, 2000]

dt = 0.005
ts = np.arange(*data_range) * dt
data = acc_ref[data_range[0]: data_range[1], 0]
# data = test_acc_ref[data_range[0]: data_range[1], 0]
ref_data = ref_filt.causal_smoother(data, m=110)

## filt

In [ ]:
def diff_gate(x, y, x_bd, y_bd, true_val, false_val, temp_x=20.0, temp_y=100.0):
    x_diff = temp_x * (x - x_bd)
    y_diff = temp_y * (y_bd - y)
    or_weight= 1.0 - (1.0 - jax.nn.sigmoid(x_diff)) * (1.0 - jax.nn.sigmoid(y_diff))
    return or_weight * true_val + (1.0 - or_weight) * false_val

In [ ]:
heur_size = 50

def cutoff_linear_filt(
    slow_freq: jax.Array,
    fast_freq: jax.Array,
    heur_cutoff: jax.Array,
    res_cutoff: jax.Array,
    temps: jax.Array,
    data: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    assert temps.shape == (2,)
    y = jnp.zeros(data.size + heur_size)  # initial padding
    yp = jnp.zeros(data.size)
    ypp = jnp.zeros(data.size)
    f = jnp.zeros(data.size)
    up = jnp.concatenate([jnp.ones(heur_size) * data[0], data])  # data (u) padded

    def filt_body(i: int, state: tuple[jax.Array, jax.Array, jax.Array, jax.Array]) -> tuple[jax.Array, jax.Array, jax.Array, jax.Array]:
        y, yp, ypp, f = state
        u_hist = jax.lax.dynamic_slice(up, [i], [heur_size])
        y_hist = jax.lax.dynamic_slice(y, [i], [heur_size])
        diff = u_hist - y_hist
        heur = jnp.square(jnp.mean(diff))
        abs_heur = jnp.mean(jnp.square(diff))
        res_heur = abs_heur - heur  # nonnegative, by Jensen
        fi = diff_gate(heur, res_heur, heur_cutoff, res_cutoff, fast_freq, slow_freq, temps[0], temps[1])
        f = f.at[i].set(fi)
        E0, E1, C0, C1, C2 = trip.fast_trip_E0_E1_C_full(fi)

        idx = i + heur_size  # index past history
        u = up[idx]  # not included in `u_hist`
        x0 = trip.fast_obs_x0(fi, y[idx - 3], y[idx - 2], y[idx - 1], up[idx - 1], up[idx - 2])
        x1 = E0 @ x0 + E1 * u
        y = y.at[idx].set(C0 @ x1)
        yp = yp.at[i].set(C1 @ x1)
        ypp = ypp.at[i].set(C2 @ x1)
        return y, yp, ypp, f

    y, yp, ypp, f = jax.lax.fori_loop(0, data.size, filt_body, (y, yp, ypp, f))
    y = y[heur_size:]  # remove initial padding
    return y, yp, ypp, f

## opt

In [ ]:
@jax.jit
def heur_cost(params: jax.Array, data: jax.Array, ref: jax.Array) -> jax.Array:
    slow_freq = -jnp.square(params[0])
    fast_freq = -jnp.square(params[1]) + slow_freq
    heur_cutoff = jnp.square(params[2])
    res_cutoff = jnp.square(params[3])
    temps = jnp.square(params[4:6])

    filt, _, filtpp, _ = cutoff_linear_filt(slow_freq, fast_freq, heur_cutoff, res_cutoff, temps, data)

    cost = jnp.mean(jnp.square(filt - ref)) * 1e2
    cost += jnp.mean(jnp.square(filtpp)) * 5e-3
    return cost

heur_cost_vg = jax.jit(jax.value_and_grad(heur_cost))

In [ ]:
heur_cutoffs = (-5.0, -10.0)
params0 = jnp.array([
    jnp.sqrt(5.0),
    jnp.sqrt(10.0 - jnp.sqrt(5.0)),
    0.2,
    0.01,
    jnp.sqrt(20.0),
    jnp.sqrt(100.0),
])
heur_cost_vg(params0, data, ref_data)

res = sci_opt.minimize(
    fun=functools.partial(heur_cost_vg, data=data, ref=ref_data),
    x0=params0,
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 10,
    }
)
res

In [ ]:
if True:
    res = sci_opt.minimize(
        fun=functools.partial(heur_cost_vg, data=data, ref=ref_data),
        x0=res.x,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 100,
        }
    )
    print(res)

## analysis

In [ ]:
sms_params = (-2 * jnp.pi, -2 * jnp.pi, 0.0, 0.0, jnp.array([20.0, 100.0]))
opt_params = (-res.x[0]**2, -(res.x[0]**2 + res.x[1]**2), res.x[2]**2, res.x[3]**2, res.x[4:6]**2)

plot_data = data
# plot_data = test_acc_ref[:, 0]
plot_ref_data = ref_filt.causal_smoother(plot_data, m=110)

opt_filt, opt_filtp, opt_filtpp, opt_fs = cutoff_linear_filt(*opt_params, data=plot_data)
sms_filt, sms_filtp, sms_filtpp, sms_fs = cutoff_linear_filt(*sms_params, data=plot_data)

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(14, 7), height_ratios=[1.0, 0.2], sharex=True)

axs[0].plot(plot_data, label="plot_data", alpha=0.2)
axs[0].plot(plot_ref_data, label="plot_ref_data", alpha=0.4)
axs[0].plot(opt_filt, label="opt_filt")
axs[0].plot(sms_filt, label="sms_filt")

axs[1].plot(opt_fs, label="fs")

for ax in axs:
    ax.legend()
    ax.grid()
fig.tight_layout()

In [ ]:
opt_params

In [ ]:
# # signed_diff = data - ref_filt
# signed_diff = data - opt_filt
# tmp_heur_size = heur_size
# heur = np.square(np.convolve(signed_diff, np.ones(tmp_heur_size) / tmp_heur_size, mode="valid"))
# abs_heur = np.convolve(np.square(signed_diff), np.ones(tmp_heur_size) / tmp_heur_size, mode="valid")
# res_heur = abs_heur - heur  # residual heuristics


# fig, ax = plt.subplots(1, 1, figsize=(14, 7))
# ax.plot(heur, label="heur")
# ax.plot(abs_heur, label="abs_heur")
# ax.plot(res_heur, label="res_heur")
# # ax.scatter(jnp.arange(fast_slow.size), fast_slow, label="fast_slow", marker="o", c="orange", s=2)
# ax.grid()
# ax.legend()